In [1]:
import warnings
warnings.filterwarnings('ignore')

import os
import pickle

import numpy as np
import pandas as pd

from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta

from pandas.tseries.offsets import MonthEnd, MonthBegin
from maricovault.MaricoDB import MaricoSnowflake

from joblib import Parallel, delayed

In [2]:
def get_dbconnection(db_name): 

    KEY_VAULT_NAME = "prod-pwd"
    if db_name == 'PROD':
        db_name = 'prod'
    else:
        db_name = 'dev'

    msf = MaricoSnowflake(KEY_VAULT_NAME)
    msf.get_db_credentials(db_name=db_name)
    msf.connect()
    dbconnection = msf.get_connection()
    
    return dbconnection

In [3]:
dev_conn = get_dbconnection('DEV')
prod_conn = get_dbconnection('PROD')


Credentials retrieved successfully for dev db.

Credentials retrieved successfully for prod db.


### Helper Functions

In [4]:
realignment_df = pd.read_sql(
    """select * from trn_mil_asm_psku_realignment""",
    dev_conn
)
realignment_df.columns = realignment_df.columns.str.lower()

def demand_driver_realign_pskus(data, channel):
    """
    Realign the old pskus to new pskus and return updated data.

    Args:
        data: pandas dataframe
        - master dataframe having all the pskus
    
    Return:
        data: pandas dataframe
        - dataframe 
    """
    realignment_data = realignment_df.copy()
    realignment_data.columns = realignment_data.columns.str.lower()
    realignment_data = realignment_data[
        (realignment_data["channel"] == channel)
        | (realignment_data["channel"] == channel + " B2C")
        | (realignment_data["channel"] == "ALL")
    ]

    data["parent_material_code"] = data["parent_material_code"].astype(int)

    for grp, grp_data in realignment_data.groupby(by=["psku old", "asm"]):
        old_psku, old_asm = grp
        new_psku = grp_data["psku new"].values[0]
        if old_asm != "ALL":
            condition = (data["parent_material_code"] == old_psku) & (
                data["asm_area_code"] == old_asm
            )
        else:
            condition = data["parent_material_code"] == old_psku

        data.loc[condition, "parent_material_code"] = new_psku

    return data

### Push heuristic data

In [ ]:
df = pd.read_excel('/data/aman_singh/acuuracy_check/All_combination_Nov25_live_with_festivals((Autorecovered-312166393861403186)).xlsb', sheet_name = 'Base')
df

,key,month_date,pred_prophet,pred_rf,pred_value_prophet,pred_value_rf,channel,asm_area_code,depot_code,parent_material_code,...,RF_Recency NS Heuristic Val,RF_Final Heursitic Val,Remark,Missing,ALL Planning Principle,NON ALL Planning Principle,Skip basis PP,Heuristic > 2x Model,Diff,Comparison
0,BCE1_D231_718589,45991,5.264959,2.7,0.000237,0.000122,ECOM,BCE1,D231,718589,...,0.000203,0.000203,NaN,0,0,0,0,0,0.0,P3M
1,BCE1_D231_718589,46022,4.547336,3.6,0.000205,0.000162,ECOM,BCE1,D231,718589,...,0.000203,0.000203,NaN,0,0,0,0,0,0.0,P3M
2,BCE1_D231_718589,46053,7.053803,2.7,0.000317,0.000122,ECOM,BCE1,D231,718589,...,0.000203,0.000203,NaN,0,0,0,0,0,0.0,P3M
3,BCE1_D231_718589,46081,5.662848,1.8,0.000255,0.000081,ECOM,BCE1,D231,718589,...,0.000203,0.000203,NaN,0,0,0,0,0,0.0,P3M
4,BCE1_D231_718589,46112,0.000000,2.7,0.000000,0.000122,ECOM,BCE1,D231,718589,...,0.000203,0.000203,NaN,0,0,0,0,1,0.0,P3M
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
164365,QCW2_D463_810125,46022,NaN,NaN,NaN,NaN,QCOM,QCW2,D463,810125,...,0.000069,0.000092,NaN,1,0,0,0,1,0.0,P3M
164366,QCW2_D463_810125,46053,NaN,NaN,NaN,NaN,QCOM,QCW2,D463,810125,...,0.000069,0.000092,NaN,1,0,0,0,1,0.0,P3M
164367,QCW2_D463_810125,46081,NaN,NaN,NaN,NaN,QCOM,QCW2,D463,810125,...,0.000069,0.000092,NaN,1,0,0,0,1,0.0,P3M
164368,QCW2_D463_810125,46112,NaN,NaN,NaN,NaN,QCOM,QCW2,D463,810125,...,0.000069,0.000092,NaN,1,0,0,0,1,0.0,P3M


In [ ]:
df['month_date'] = (
    pd.to_datetime(df['month_date'], unit='D', origin='1899-12-30')
      .dt.to_period('M')
      .dt.to_timestamp()
)
df['run_month'] = (
    pd.to_datetime(df['run_month'], unit='D', origin='1899-12-30')
      .dt.to_period('M')
      .dt.to_timestamp()
)

df['month_date'] = (
    pd.to_datetime(df['month_date']) + pd.offsets.MonthEnd(0)
)
df['run_month'] = (
    pd.to_datetime(df['run_month']) + pd.offsets.MonthEnd(0)
)

In [ ]:
df['run_month'].unique()

<DatetimeArray>
['2025-11-30 00:00:00']
Length: 1, dtype: datetime64[ns]

In [ ]:
df.columns

Index(['key', 'month_date', 'pred_prophet', 'pred_rf', 'pred_value_prophet',
       'pred_value_rf', 'channel', 'asm_area_code', 'depot_code',
       'parent_material_code', 'brand_code', 'sec_vol_actuals_rum_month_value',
       'pred_best_model', 'pred_value_best_model',
       'sec_vol_actuals_rum_month_treated',
       'sec_vol_actuals_rum_month_value_treated', 'train_till', 'cov',
       'run_month', 'M month', 'pred_prophet_70%ile', 'portfolio',
       'qtr_ind_rate', 'sec_vol_actuals_rum_month', 'P3M', 'P6M', 'LY P3M',
       'LY P6M', 'LY P3M_copy', 'P3M Max', 'P3M Top 2 Mean', 'MoM P3M growth',
       'MoM P3M growth_lag_1', 'MoM P3M growth_lag_2', 'Growth Flag',
       '>=20%_3M_inc_month_count', 'Avg(P3M Mean, Max)', 'P3M_value',
       'P6M_value', 'LY P3M_value', 'LY P6M_value',
       'pred_prophet_70%ile_value', 'LY', 'LLY', 'LY value', 'LLY value',
       'Sec_Value_in_Cr_lag_1', 'Sec_Value_in_Cr_lag_2',
       'Sec_Value_in_Cr_lag_3', 'ASM', 'Depot', 'PSKU', 'class', '

In [ ]:
df['month_date'] = df['month_date'].astype(str)
df['run_month'] = df['run_month'].astype(str)
df.columns

Index(['key', 'month_date', 'pred_prophet', 'pred_rf', 'pred_value_prophet',
       'pred_value_rf', 'channel', 'asm_area_code', 'depot_code',
       'parent_material_code', 'brand_code', 'sec_vol_actuals_rum_month_value',
       'pred_best_model', 'pred_value_best_model',
       'sec_vol_actuals_rum_month_treated',
       'sec_vol_actuals_rum_month_value_treated', 'train_till', 'cov',
       'run_month', 'M month', 'pred_prophet_70%ile', 'portfolio',
       'qtr_ind_rate', 'sec_vol_actuals_rum_month', 'P3M', 'P6M', 'LY P3M',
       'LY P6M', 'LY P3M_copy', 'P3M Max', 'P3M Top 2 Mean', 'MoM P3M growth',
       'MoM P3M growth_lag_1', 'MoM P3M growth_lag_2', 'Growth Flag',
       '>=20%_3M_inc_month_count', 'Avg(P3M Mean, Max)', 'P3M_value',
       'P6M_value', 'LY P3M_value', 'LY P6M_value',
       'pred_prophet_70%ile_value', 'LY', 'LLY', 'LY value', 'LLY value',
       'Sec_Value_in_Cr_lag_1', 'Sec_Value_in_Cr_lag_2',
       'Sec_Value_in_Cr_lag_3', 'ASM', 'Depot', 'PSKU', 'class', '

In [ ]:
upload_df = df[['channel','portfolio', 'brand_code', 'class',
        'run_month', 'M month','month_date', 'ASM', 'Depot', 'PSKU','pred_prophet', 'pred_rf','Final Heuristic 2 Vol',
        'RF_Final Heuristic Vol']].rename(
            columns = {'brand_code':'brand', 'class':'Brand Class', 'month_date':'month','pred_prophet':'prophet vol',
                       'pred_rf':'rf_vol', 'Final Heuristic 2 Vol':'prophet heuristic vol',
                       'RF_Final Heuristic Vol':'rf heuristic vol'}
        )
upload_df

,channel,portfolio,brand,Brand Class,run_month,M month,month,ASM,Depot,PSKU,prophet vol,rf_vol,prophet heuristic vol,rf heuristic vol
0,ECOM,Hair Oils,ADV-AHO-R,B,2025-11-30,M,2025-11-30,BCE1,D231,718589,5.264959,2.7,5.264959,4.50
1,ECOM,Hair Oils,ADV-AHO-R,B,2025-11-30,M+1,2025-12-31,BCE1,D231,718589,4.547336,3.6,4.547336,4.50
2,ECOM,Hair Oils,ADV-AHO-R,B,2025-11-30,M+2,2026-01-31,BCE1,D231,718589,7.053803,2.7,7.053803,4.50
3,ECOM,Hair Oils,ADV-AHO-R,B,2025-11-30,M+3,2026-02-28,BCE1,D231,718589,5.662848,1.8,5.662848,4.50
4,ECOM,Hair Oils,ADV-AHO-R,B,2025-11-30,M+4,2026-03-31,BCE1,D231,718589,0.000000,2.7,4.500000,4.50
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
164365,QCOM,Male Grooming,SW_SGPRF,C,2025-11-30,M+1,2025-12-31,QCW2,D463,810125,NaN,NaN,0.640000,0.64
164366,QCOM,Male Grooming,SW_SGPRF,C,2025-11-30,M+2,2026-01-31,QCW2,D463,810125,NaN,NaN,0.640000,0.64
164367,QCOM,Male Grooming,SW_SGPRF,C,2025-11-30,M+3,2026-02-28,QCW2,D463,810125,NaN,NaN,0.640000,0.64
164368,QCOM,Male Grooming,SW_SGPRF,C,2025-11-30,M+4,2026-03-31,QCW2,D463,810125,NaN,NaN,0.640000,0.64


In [ ]:
upload_df.columns = upload_df.columns.str.upper()
upload_df

,CHANNEL,PORTFOLIO,BRAND,BRAND CLASS,RUN_MONTH,M MONTH,MONTH,ASM,DEPOT,PSKU,PROPHET VOL,RF_VOL,PROPHET HEURISTIC VOL,RF HEURISTIC VOL
0,ECOM,Hair Oils,ADV-AHO-R,B,2025-11-30,M,2025-11-30,BCE1,D231,718589,5.264959,2.7,5.264959,4.50
1,ECOM,Hair Oils,ADV-AHO-R,B,2025-11-30,M+1,2025-12-31,BCE1,D231,718589,4.547336,3.6,4.547336,4.50
2,ECOM,Hair Oils,ADV-AHO-R,B,2025-11-30,M+2,2026-01-31,BCE1,D231,718589,7.053803,2.7,7.053803,4.50
3,ECOM,Hair Oils,ADV-AHO-R,B,2025-11-30,M+3,2026-02-28,BCE1,D231,718589,5.662848,1.8,5.662848,4.50
4,ECOM,Hair Oils,ADV-AHO-R,B,2025-11-30,M+4,2026-03-31,BCE1,D231,718589,0.000000,2.7,4.500000,4.50
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
164365,QCOM,Male Grooming,SW_SGPRF,C,2025-11-30,M+1,2025-12-31,QCW2,D463,810125,NaN,NaN,0.640000,0.64
164366,QCOM,Male Grooming,SW_SGPRF,C,2025-11-30,M+2,2026-01-31,QCW2,D463,810125,NaN,NaN,0.640000,0.64
164367,QCOM,Male Grooming,SW_SGPRF,C,2025-11-30,M+3,2026-02-28,QCW2,D463,810125,NaN,NaN,0.640000,0.64
164368,QCOM,Male Grooming,SW_SGPRF,C,2025-11-30,M+4,2026-03-31,QCW2,D463,810125,NaN,NaN,0.640000,0.64


In [ ]:
from snowflake.connector.pandas_tools import write_pandas

write_pandas(dev_conn, upload_df, 
            table_name = "TRN_MIL_DF_HEURISTICS_OUTPUT",
            auto_create_table=True,
            overwrite = False,)

(True,
 1,
 164370,
 [('rtthbeires/file0.txt',
   'LOADED',
   164370,
   164370,
   1,
   0,
   None,
   None,
   None,
   None)])

### Push shared output

In [20]:
df = pd.read_excel(
    '/data/aman_singh/acuuracy_check/Stat Demand Forecast GT_as_on_11th_Mar_2026.xlsb',
    sheet_name='Base'
)

In [21]:
df['run_month'] = '2026-03-31'
df

,Channel,Sub Channel,Portfolio,Brand,Brand Class,Qtr Index Rate,Key,Month,ASM,Depot,...,Planning Principle,LY Val (Cr),P3M Val (Cr),P6M Val (Cr),LY P3M Val (Cr),LY P6M Val (Cr),LY P3M (Rolling) Val (Cr),Pred Val (Cr),Primary P3M 0?,run_month
0,GT,NaN,Saffola Oils,SAFF GOLD,A,137662.938527,AURG_D3A4_718288,46142,AURG,D3A4,...,valid,0.045153,0.060870,0.058300,0.050912,0.046186,0.045543,0.063553,False,2026-03-31
1,GT,NaN,Saffola Oils,SAFF GOLD,A,137662.938527,AURG_D3A4_718288,46173,AURG,D3A4,...,valid,0.048526,0.060870,0.058300,0.050912,0.044190,0.039830,0.063553,False,2026-03-31
2,GT,NaN,Saffola Oils,SAFF GOLD,A,137662.938527,AURG_D3A4_718288,46203,AURG,D3A4,...,valid,0.052587,0.060870,0.058300,0.050912,0.047517,0.044121,0.063553,False,2026-03-31
3,GT,NaN,Saffola Oils,SAFF GOLD,A,137662.938527,AURG_D3A4_718288,46234,AURG,D3A4,...,valid,0.064426,0.060870,0.058300,0.050912,0.047150,0.048756,0.063553,False,2026-03-31
4,GT,NaN,CNO,PCNO(R),A,309765.865129,AURG_D3A4_718297,46142,AURG,D3A4,...,valid,0.542896,0.700463,0.616672,0.382406,0.356463,0.266461,0.795640,False,2026-03-31
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33967,GT,NaN,Foods,SAF_HONEY,C,215312.053116,WUP_D113_808731,46234,WUP,D113,...,valid,0.000129,0.000000,0.000000,0.000129,0.000022,0.000000,0.000000,True,2026-03-31
33968,GT,NaN,Foods,SAF_HONEY,C,215312.053116,WUP_D113_809238,46142,WUP,D113,...,valid,0.000000,0.000014,0.000007,0.000000,0.000000,0.000000,0.000000,False,2026-03-31
33969,GT,NaN,Foods,SAF_HONEY,C,215312.053116,WUP_D113_809238,46173,WUP,D113,...,valid,0.000000,0.000014,0.000007,0.000000,0.000000,0.000000,0.000000,False,2026-03-31
33970,GT,NaN,Foods,SAF_HONEY,C,215312.053116,WUP_D113_809238,46203,WUP,D113,...,valid,0.000000,0.000014,0.000007,0.000000,0.000000,0.000000,0.000000,False,2026-03-31


In [22]:
df['Month'] = (
    pd.to_datetime(df['Month'], unit='D', origin='1899-12-30')
      .dt.to_period('M')
      .dt.to_timestamp()
)
df

,Channel,Sub Channel,Portfolio,Brand,Brand Class,Qtr Index Rate,Key,Month,ASM,Depot,...,Planning Principle,LY Val (Cr),P3M Val (Cr),P6M Val (Cr),LY P3M Val (Cr),LY P6M Val (Cr),LY P3M (Rolling) Val (Cr),Pred Val (Cr),Primary P3M 0?,run_month
0,GT,NaN,Saffola Oils,SAFF GOLD,A,137662.938527,AURG_D3A4_718288,2026-04-01,AURG,D3A4,...,valid,0.045153,0.060870,0.058300,0.050912,0.046186,0.045543,0.063553,False,2026-03-31
1,GT,NaN,Saffola Oils,SAFF GOLD,A,137662.938527,AURG_D3A4_718288,2026-05-01,AURG,D3A4,...,valid,0.048526,0.060870,0.058300,0.050912,0.044190,0.039830,0.063553,False,2026-03-31
2,GT,NaN,Saffola Oils,SAFF GOLD,A,137662.938527,AURG_D3A4_718288,2026-06-01,AURG,D3A4,...,valid,0.052587,0.060870,0.058300,0.050912,0.047517,0.044121,0.063553,False,2026-03-31
3,GT,NaN,Saffola Oils,SAFF GOLD,A,137662.938527,AURG_D3A4_718288,2026-07-01,AURG,D3A4,...,valid,0.064426,0.060870,0.058300,0.050912,0.047150,0.048756,0.063553,False,2026-03-31
4,GT,NaN,CNO,PCNO(R),A,309765.865129,AURG_D3A4_718297,2026-04-01,AURG,D3A4,...,valid,0.542896,0.700463,0.616672,0.382406,0.356463,0.266461,0.795640,False,2026-03-31
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33967,GT,NaN,Foods,SAF_HONEY,C,215312.053116,WUP_D113_808731,2026-07-01,WUP,D113,...,valid,0.000129,0.000000,0.000000,0.000129,0.000022,0.000000,0.000000,True,2026-03-31
33968,GT,NaN,Foods,SAF_HONEY,C,215312.053116,WUP_D113_809238,2026-04-01,WUP,D113,...,valid,0.000000,0.000014,0.000007,0.000000,0.000000,0.000000,0.000000,False,2026-03-31
33969,GT,NaN,Foods,SAF_HONEY,C,215312.053116,WUP_D113_809238,2026-05-01,WUP,D113,...,valid,0.000000,0.000014,0.000007,0.000000,0.000000,0.000000,0.000000,False,2026-03-31
33970,GT,NaN,Foods,SAF_HONEY,C,215312.053116,WUP_D113_809238,2026-06-01,WUP,D113,...,valid,0.000000,0.000014,0.000007,0.000000,0.000000,0.000000,0.000000,False,2026-03-31


In [23]:
df.columns

Index(['Channel', 'Sub Channel', 'Portfolio', 'Brand', 'Brand Class',
       'Qtr Index Rate', 'Key', 'Month', 'ASM', 'Depot', 'PSKU',
       'LY Vol (ROUM)', 'P3M Vol (ROUM)', 'P6M Vol (ROUM)',
       'LY P3M Vol (ROUM)', 'LY P6M Vol (ROUM)', 'LY P3M (Rolling) Vol (ROUM)',
       'Pred Vol (ROUM)', 'Sec Value Lag 1 (Cr)', 'Sec Value Lag 2 (Cr)',
       'Sec Value Lag 3 (Cr)', 'LY Value Lag 1 (Cr)', 'LY Value Lag 2 (Cr)',
       'LY Value Lead 1 (Cr)', 'LY Value Lead 2 (Cr)', 'Planning Principle',
       'LY Val (Cr)', 'P3M Val (Cr)', 'P6M Val (Cr)', 'LY P3M Val (Cr)',
       'LY P6M Val (Cr)', 'LY P3M (Rolling) Val (Cr)', 'Pred Val (Cr)',
       'Primary P3M 0?', 'run_month'],
      dtype='object')

In [ ]:
#df.drop(['Qtr Index Rate', 'Brand Class', 'LY (ROUM)', 'LY Value (in Cr)', 'Pred Value (in Cr)'], axis=1, inplace=True)

In [24]:
df['Month'] = (
    pd.to_datetime(df['Month']) + pd.offsets.MonthEnd(0)
)

In [25]:
df['run_month'] = pd.to_datetime(df['run_month'])
df['Month'] = pd.to_datetime(df['Month'])


mappings = {}

for run_month in df['run_month'].unique():
    # run_month = pd.to_datetime(run_month)

    if not run_month in mappings:
        mappings[run_month] = {}

    mappings[run_month][run_month] = 'M'

    for idx in range(1, 9):
        mappings[run_month][run_month + MonthEnd(idx)] = f"M+{idx}"


mappings   

{Timestamp('2026-03-31 00:00:00'): {Timestamp('2026-03-31 00:00:00'): 'M',
  Timestamp('2026-04-30 00:00:00'): 'M+1',
  Timestamp('2026-05-31 00:00:00'): 'M+2',
  Timestamp('2026-06-30 00:00:00'): 'M+3',
  Timestamp('2026-07-31 00:00:00'): 'M+4',
  Timestamp('2026-08-31 00:00:00'): 'M+5',
  Timestamp('2026-09-30 00:00:00'): 'M+6',
  Timestamp('2026-10-31 00:00:00'): 'M+7',
  Timestamp('2026-11-30 00:00:00'): 'M+8'}}

In [26]:
df['M month'] = df.apply(
    lambda x: mappings[x['run_month']].get(
        x['Month']
    ), axis=1
)

In [27]:
df

,Channel,Sub Channel,Portfolio,Brand,Brand Class,Qtr Index Rate,Key,Month,ASM,Depot,...,LY Val (Cr),P3M Val (Cr),P6M Val (Cr),LY P3M Val (Cr),LY P6M Val (Cr),LY P3M (Rolling) Val (Cr),Pred Val (Cr),Primary P3M 0?,run_month,M month
0,GT,NaN,Saffola Oils,SAFF GOLD,A,137662.938527,AURG_D3A4_718288,2026-04-30,AURG,D3A4,...,0.045153,0.060870,0.058300,0.050912,0.046186,0.045543,0.063553,False,2026-03-31,M+1
1,GT,NaN,Saffola Oils,SAFF GOLD,A,137662.938527,AURG_D3A4_718288,2026-05-31,AURG,D3A4,...,0.048526,0.060870,0.058300,0.050912,0.044190,0.039830,0.063553,False,2026-03-31,M+2
2,GT,NaN,Saffola Oils,SAFF GOLD,A,137662.938527,AURG_D3A4_718288,2026-06-30,AURG,D3A4,...,0.052587,0.060870,0.058300,0.050912,0.047517,0.044121,0.063553,False,2026-03-31,M+3
3,GT,NaN,Saffola Oils,SAFF GOLD,A,137662.938527,AURG_D3A4_718288,2026-07-31,AURG,D3A4,...,0.064426,0.060870,0.058300,0.050912,0.047150,0.048756,0.063553,False,2026-03-31,M+4
4,GT,NaN,CNO,PCNO(R),A,309765.865129,AURG_D3A4_718297,2026-04-30,AURG,D3A4,...,0.542896,0.700463,0.616672,0.382406,0.356463,0.266461,0.795640,False,2026-03-31,M+1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33967,GT,NaN,Foods,SAF_HONEY,C,215312.053116,WUP_D113_808731,2026-07-31,WUP,D113,...,0.000129,0.000000,0.000000,0.000129,0.000022,0.000000,0.000000,True,2026-03-31,M+4
33968,GT,NaN,Foods,SAF_HONEY,C,215312.053116,WUP_D113_809238,2026-04-30,WUP,D113,...,0.000000,0.000014,0.000007,0.000000,0.000000,0.000000,0.000000,False,2026-03-31,M+1
33969,GT,NaN,Foods,SAF_HONEY,C,215312.053116,WUP_D113_809238,2026-05-31,WUP,D113,...,0.000000,0.000014,0.000007,0.000000,0.000000,0.000000,0.000000,False,2026-03-31,M+2
33970,GT,NaN,Foods,SAF_HONEY,C,215312.053116,WUP_D113_809238,2026-06-30,WUP,D113,...,0.000000,0.000014,0.000007,0.000000,0.000000,0.000000,0.000000,False,2026-03-31,M+3


In [28]:
df['Month'] = df['Month'].astype(str)
df['run_month'] = df['run_month'].astype(str)
df.columns

Index(['Channel', 'Sub Channel', 'Portfolio', 'Brand', 'Brand Class',
       'Qtr Index Rate', 'Key', 'Month', 'ASM', 'Depot', 'PSKU',
       'LY Vol (ROUM)', 'P3M Vol (ROUM)', 'P6M Vol (ROUM)',
       'LY P3M Vol (ROUM)', 'LY P6M Vol (ROUM)', 'LY P3M (Rolling) Vol (ROUM)',
       'Pred Vol (ROUM)', 'Sec Value Lag 1 (Cr)', 'Sec Value Lag 2 (Cr)',
       'Sec Value Lag 3 (Cr)', 'LY Value Lag 1 (Cr)', 'LY Value Lag 2 (Cr)',
       'LY Value Lead 1 (Cr)', 'LY Value Lead 2 (Cr)', 'Planning Principle',
       'LY Val (Cr)', 'P3M Val (Cr)', 'P6M Val (Cr)', 'LY P3M Val (Cr)',
       'LY P6M Val (Cr)', 'LY P3M (Rolling) Val (Cr)', 'Pred Val (Cr)',
       'Primary P3M 0?', 'run_month', 'M month'],
      dtype='object')

In [29]:
upload_df = df[['Channel','Portfolio', 'Brand', 'Brand Class',
        'run_month', 'M month','Month', 'ASM', 'Depot', 'PSKU','Pred Vol (ROUM)']]

In [30]:
upload_df

,Channel,Portfolio,Brand,Brand Class,run_month,M month,Month,ASM,Depot,PSKU,Pred Vol (ROUM)
0,GT,Saffola Oils,SAFF GOLD,A,2026-03-31,M+1,2026-04-30,AURG,D3A4,718288,4.616561
1,GT,Saffola Oils,SAFF GOLD,A,2026-03-31,M+2,2026-05-31,AURG,D3A4,718288,4.616561
2,GT,Saffola Oils,SAFF GOLD,A,2026-03-31,M+3,2026-06-30,AURG,D3A4,718288,4.616561
3,GT,Saffola Oils,SAFF GOLD,A,2026-03-31,M+4,2026-07-31,AURG,D3A4,718288,4.616561
4,GT,CNO,PCNO(R),A,2026-03-31,M+1,2026-04-30,AURG,D3A4,718297,25.685215
...,...,...,...,...,...,...,...,...,...,...,...
33967,GT,Foods,SAF_HONEY,C,2026-03-31,M+4,2026-07-31,WUP,D113,808731,0.000000
33968,GT,Foods,SAF_HONEY,C,2026-03-31,M+1,2026-04-30,WUP,D113,809238,0.000000
33969,GT,Foods,SAF_HONEY,C,2026-03-31,M+2,2026-05-31,WUP,D113,809238,0.000000
33970,GT,Foods,SAF_HONEY,C,2026-03-31,M+3,2026-06-30,WUP,D113,809238,0.000000


In [ ]:
# upload_df[upload_df['Month'] == '2026-04-30']['Pred Vol (ROUM)'].sum()

5346018.0036901245

In [33]:
upload_df.columns = upload_df.columns.str.upper()
upload_df

,CHANNEL,PORTFOLIO,BRAND,BRAND CLASS,RUN_MONTH,M MONTH,MONTH,ASM,DEPOT,PSKU,PRED VOL (ROUM)
0,GT,Saffola Oils,SAFF GOLD,A,2026-03-31,M+1,2026-04-30,AURG,D3A4,718288,4.616561
1,GT,Saffola Oils,SAFF GOLD,A,2026-03-31,M+2,2026-05-31,AURG,D3A4,718288,4.616561
2,GT,Saffola Oils,SAFF GOLD,A,2026-03-31,M+3,2026-06-30,AURG,D3A4,718288,4.616561
3,GT,Saffola Oils,SAFF GOLD,A,2026-03-31,M+4,2026-07-31,AURG,D3A4,718288,4.616561
4,GT,CNO,PCNO(R),A,2026-03-31,M+1,2026-04-30,AURG,D3A4,718297,25.685215
...,...,...,...,...,...,...,...,...,...,...,...
33967,GT,Foods,SAF_HONEY,C,2026-03-31,M+4,2026-07-31,WUP,D113,808731,0.000000
33968,GT,Foods,SAF_HONEY,C,2026-03-31,M+1,2026-04-30,WUP,D113,809238,0.000000
33969,GT,Foods,SAF_HONEY,C,2026-03-31,M+2,2026-05-31,WUP,D113,809238,0.000000
33970,GT,Foods,SAF_HONEY,C,2026-03-31,M+3,2026-06-30,WUP,D113,809238,0.000000


In [34]:
# push data to snowflake
from snowflake.connector.pandas_tools import write_pandas

write_pandas(dev_conn, upload_df, 
            table_name = "TRN_MIL_DF_SHARED",
            auto_create_table=True,
            overwrite = False,)

(True,
 1,
 33972,
 [('frmfewppee/file0.txt',
   'LOADED',
   33972,
   33972,
   1,
   0,
   None,
   None,
   None,
   None)])

### push offtakes to primary

In [57]:
qcom_df = pd.read_excel(
    '/data/aman_singh/acuuracy_check/Ecom Depot Psku Primary_as_on_10th_mar.xlsx',
    sheet_name='Base'
)

In [58]:
qcom_df

,Key,Depot,PSKU,PSKU Description,Brand,Run Month,Month Date,Calculated PSKU Primary Vol,Primary Actuals Vol,Primary Plan Vol,...,LY Primary Actuals Lag 3 Val,LY Primary Actuals Lead 1 Val,LY Primary Actuals Lead 2 Val,LY Primary P3M Val,PSKU Primary P3M Sum Val,Calculated Depot PSKU Primary Val,M Month,Brand Class,Planning Principle,Primary P3M 0?
0,D111_709567,D111,709567,SAF OATS 400G(FRE WT SAF OATS 1KG P)-N,SAFF OATS,2026-03-31,2026-04-30,NaN,0,0,...,0.0,0.0,0.0,0.0,0.000000,0.0,M+1,A,Valid,True
1,D111_709567,D111,709567,SAF OATS 400G(FRE WT SAF OATS 1KG P)-N,SAFF OATS,2026-03-31,2026-05-31,NaN,0,0,...,0.0,0.0,0.0,0.0,0.000000,0.0,M+2,A,Valid,True
2,D111_709567,D111,709567,SAF OATS 400G(FRE WT SAF OATS 1KG P)-N,SAFF OATS,2026-03-31,2026-06-30,NaN,0,0,...,0.0,0.0,0.0,0.0,0.000000,0.0,M+3,A,Valid,True
3,D111_709567,D111,709567,SAF OATS 400G(FRE WT SAF OATS 1KG P)-N,SAFF OATS,2026-03-31,2026-07-31,NaN,0,0,...,0.0,0.0,0.0,0.0,0.000000,0.0,M+4,A,Valid,True
4,D111_718288,D111,718288,SAFF GOLD 5L JAR,SAFF GOLD,2026-03-31,2026-04-30,43.757146,0,0,...,0.0,0.0,0.0,0.0,0.940128,0.0,M+1,A,Valid,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39327,D677_811169,D677,811169,SW MINI EDP GIFT PO4 18ML,SW_SGPRF,2026-03-31,2026-07-31,0.000000,0,0,...,NaN,NaN,NaN,NaN,0.000000,0.0,M+4,C,Valid,True
39328,D677_811181,D677,811181,SAFFOLA COLDPRESS CNO 1L,SAF_CDPRS,2026-03-31,2026-04-30,0.003355,0,0,...,NaN,NaN,NaN,NaN,0.129492,0.0,M+1,C,Valid,True
39329,D677_811181,D677,811181,SAFFOLA COLDPRESS CNO 1L,SAF_CDPRS,2026-03-31,2026-05-31,0.004645,0,0,...,NaN,NaN,NaN,NaN,0.129492,0.0,M+2,C,Valid,True
39330,D677_811181,D677,811181,SAFFOLA COLDPRESS CNO 1L,SAF_CDPRS,2026-03-31,2026-06-30,0.004355,0,0,...,NaN,NaN,NaN,NaN,0.129492,0.0,M+3,C,Valid,True


In [37]:
qcom_df['Month Date'] = (
    pd.to_datetime(qcom_df['Month Date'], unit='D', origin='1899-12-30')
      .dt.to_period('M')
      .dt.to_timestamp()
)
qcom_df

ValueError: '0       2026-04-30
1       2026-05-31
2       2026-06-30
3       2026-07-31
4       2026-04-30
           ...    
41063   2026-07-31
41064   2026-04-30
41065   2026-05-31
41066   2026-06-30
41067   2026-07-31
Name: Month Date, Length: 41068, dtype: datetime64[ns]' is not compatible with origin='1899-12-30'; it must be numeric with a unit specified

In [57]:
qcom_df['Month Date'] = (
    pd.to_datetime(qcom_df['Month Date']) + pd.offsets.MonthEnd(0)
)

In [58]:
qcom_df['Run Month'] = (
    pd.to_datetime(qcom_df['Run Month'], unit='D', origin='1899-12-30')
      .dt.to_period('M')
      .dt.to_timestamp()
)
qcom_df['Run Month'] = (
    pd.to_datetime(qcom_df['Run Month']) + pd.offsets.MonthEnd(0)
)
qcom_df

,Key,Depot,PSKU,PSKU Desc,Brand,Portfolio,Index Rate,Run Month,Month Date,M Month,...,LY Offtake Chain FC PSKU Lead 2 Val,Actual Closing SOH Val,Actual Closing SOH Lag 1 Val,Actual Closing SOH Lag 2 Val,Final Assumed Closing SOH Val,Final Assumed Closing SOH Lag 1 Val,Safety Stock Val,Brand Class,Planning Principle,Primary P3M 0?
0,D112_709567,D112,709567,SAF OATS 400G(FRE WT SAF OATS 1KG P)-N,SAFF OATS,Foods,126480.737807,2026-02-28,2026-03-31,M+1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,A,Valid,True
1,D112_709567,D112,709567,SAF OATS 400G(FRE WT SAF OATS 1KG P)-N,SAFF OATS,Foods,126480.737807,2026-02-28,2026-04-30,M+2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,A,Valid,True
2,D112_709567,D112,709567,SAF OATS 400G(FRE WT SAF OATS 1KG P)-N,SAFF OATS,Foods,126480.737807,2026-02-28,2026-05-31,M+3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,A,Valid,True
3,D112_709567,D112,709567,SAF OATS 400G(FRE WT SAF OATS 1KG P)-N,SAFF OATS,Foods,126480.737807,2026-02-28,2026-06-30,M+4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,A,Valid,True
4,D112_718287,D112,718287,PCNO 200ml JAR,PCNO(R),CNO,309765.865129,2026-02-28,2026-03-31,M+1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,A,Valid,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30603,D677_810805,D677,810805,PA BABY FACE BODY WIPES 1KG,PABABY_GM,Skin Care,451.133000,2026-02-28,2026-06-30,M+4,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,C,Valid,True
30604,D677_811181,D677,811181,SAFFOLA COLDPRESS CNO 1L,SAF_CDPRS,Saffola Oils,330000.000000,2026-02-28,2026-03-31,M+1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,C,Valid,True
30605,D677_811181,D677,811181,SAFFOLA COLDPRESS CNO 1L,SAF_CDPRS,Saffola Oils,330000.000000,2026-02-28,2026-04-30,M+2,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,C,Valid,True
30606,D677_811181,D677,811181,SAFFOLA COLDPRESS CNO 1L,SAF_CDPRS,Saffola Oils,330000.000000,2026-02-28,2026-05-31,M+3,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,C,Valid,True


In [59]:
qcom_df['Run Month'] = pd.to_datetime(qcom_df['Run Month'])
qcom_df['Month Date'] = pd.to_datetime(qcom_df['Month Date'])

mappings ={}

for run_month in qcom_df['Run Month'].unique():
    # run_month = pd.to_datetime(run_month)

    if not run_month in mappings:
        mappings[run_month] = {}

    mappings[run_month][run_month] = 'M'

    for idx in range(1, 9):
        mappings[run_month][run_month + MonthEnd(idx)] = f"M+{idx}"


   
qcom_df['M month'] = qcom_df.apply(
    lambda x: mappings[x['Run Month']].get(
        x['Month Date']
    ), axis=1
)

In [ ]:
# qcom_df = qcom_df[qcom_df['M month'] == 'M+1']
# qcom_df

,key2,key,Chain,FC,PSKU,PSKU Desc,Brand,Index Rate,Portfolio,Run Month,...,Offtake Chain FC PSKU Lag 3 Vol,LY Offtake Actuals Chain FC PSKU Vol,LY Offtake Chain FC PSKU P3M Vol,Offtake Chain FC PSKU P3M Vol,Offtake Chain FC PSKU Lag 1 Val,Offtake Chain FC PSKU Lag 2 Val,Offtake Chain FC PSKU Lag 3 Val,LY Offtake Actuals Chain FC PSKU Val,LY Offtake Chain FC PSKU P3M Val,Offtake Chain FC PSKU P3M Val
2,Blinkit_ahmedabad a2 - feeder warehouse_718288,Blinkit_718288,Blinkit,ahmedabad a2 - feeder warehouse,718288,SAFF GOLD 5L JAR,SAFF GOLD,137662.938527,Saffola Oils,2025-11-30,...,6.873931,5.219777,3.804556,6.602009,0.107562,0.085510,0.094629,0.071857,0.052375,0.090885
9,Blinkit_ahmedabad a2 - feeder warehouse_718299,Blinkit_718299,Blinkit,ahmedabad a2 - feeder warehouse,718299,PCNO 100ml BTL,PCNO(R),309765.865129,CNO,2025-11-30,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
16,Blinkit_ahmedabad a2 - feeder warehouse_718308,Blinkit_718308,Blinkit,ahmedabad a2 - feeder warehouse,718308,PCNO 100ml JAR,PCNO(R),309765.865129,CNO,2025-11-30,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
23,Blinkit_ahmedabad a2 - feeder warehouse_718310,Blinkit_718310,Blinkit,ahmedabad a2 - feeder warehouse,718310,PCNO 500ml JAR,PCNO(R),309765.865129,CNO,2025-11-30,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
30,Blinkit_ahmedabad a2 - feeder warehouse_718312,Blinkit_718312,Blinkit,ahmedabad a2 - feeder warehouse,718312,PCNO 1L JAR,PCNO(R),309765.865129,CNO,2025-11-30,...,0.196489,0.114393,0.065902,0.201895,0.006292,0.005917,0.006087,0.003543,0.002041,0.006254
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
235825,Zepto_pun-dry-mh2-koregaon_810673,Zepto_810673,Zepto,pun-dry-mh2-koregaon,810673,PA ESS ROSEMARY OIL 14ML,PA_ESS_HO,12900.000000,Hair Oils,2025-11-30,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
235832,Zepto_pun-dry-mh2-koregaon_810674,Zepto_810674,Zepto,pun-dry-mh2-koregaon,810674,PA ESS TEA TREE OIL 14ML,PA_ESS_HO,12900.000000,Hair Oils,2025-11-30,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
235839,Zepto_pun-dry-mh2-koregaon_810685,Zepto_810685,Zepto,pun-dry-mh2-koregaon,810685,SF MUESLI MANGO 400G POUCH,SAF-MUSLI,321959.667548,Foods,2025-11-30,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
235846,Zepto_pun-dry-mh2-koregaon_810738,Zepto_810738,Zepto,pun-dry-mh2-koregaon,810738,PA BABY FACE BODY WIPE 362GM,PABABY_GM,451.133000,Skin Care,2025-11-30,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [60]:
qcom_df.columns

Index(['Key', 'Depot', 'PSKU', 'PSKU Description', 'Brand', 'Run Month',
       'Month Date', 'Calculated PSKU Primary Vol', 'Primary Actuals Vol',
       'Primary Plan Vol', 'Secondary Plan Vol', 'Secondary Actuals Vol',
       'Primary P3M Vol', 'Primary Actuals Lag 1 Vol',
       'Primary Actuals Lag 2 Vol', 'Primary Actuals Lag 3 Vol',
       'LY Primary Actuals Vol', 'LY Primary Actuals Lag 1 Vol',
       'LY Primary Actuals Lag 2 Vol', 'LY Primary Actuals Lag 3 Vol',
       'LY Primary Actuals Lead 1 Vol', 'LY Primary Actuals Lead 2 Vol',
       'LY Primary P3M Vol', 'Primary P3M copy Vol',
       'PSKU Primary P3M Sum Vol', 'PSKU P3M Contribution',
       'Calculated Depot PSKU Primary Vol', 'Index Rate',
       'Calculated PSKU Primary Val', 'Primary Actuals Val',
       'Primary Plan Val', 'Secondary Plan Val', 'Secondary Actuals Val',
       'Primary P3M Val', 'Primary Actuals Lag 1 Val',
       'Primary Actuals Lag 2 Val', 'Primary Actuals Lag 3 Val',
       'LY Primary Actu

In [61]:
qcom_df.rename(columns = {'Calculated Depot PSKU Primary Vol':'Calculated Primary Vol'}, inplace = True)

In [62]:
brand_md_df = pd.read_excel(r"/data/aman_singh/mt_forecast/Brand_metadata.xlsx")
brand_md_df.rename(columns = {'brand_code':'Brand'}, inplace = True)

In [63]:

len_before_merge = len(qcom_df)
qcom_df = qcom_df.merge(
    brand_md_df,
    on=['Brand'],
    how='left'
)
assert len_before_merge == len(qcom_df)

In [64]:
qcom_df

,Key,Depot,PSKU,PSKU Description,Brand,Run Month,Month Date,Calculated PSKU Primary Vol,Primary Actuals Vol,Primary Plan Vol,...,LY Primary Actuals Lead 2 Val,LY Primary P3M Val,PSKU Primary P3M Sum Val,Calculated Depot PSKU Primary Val,M Month,Brand Class,Planning Principle,Primary P3M 0?,M month,portfolio
0,D111_709567,D111,709567,SAF OATS 400G(FRE WT SAF OATS 1KG P)-N,SAFF OATS,2026-03-31,2026-04-30,NaN,0,0,...,0.0,0.0,0.000000,0.0,M+1,A,Valid,True,M+1,Foods
1,D111_709567,D111,709567,SAF OATS 400G(FRE WT SAF OATS 1KG P)-N,SAFF OATS,2026-03-31,2026-05-31,NaN,0,0,...,0.0,0.0,0.000000,0.0,M+2,A,Valid,True,M+2,Foods
2,D111_709567,D111,709567,SAF OATS 400G(FRE WT SAF OATS 1KG P)-N,SAFF OATS,2026-03-31,2026-06-30,NaN,0,0,...,0.0,0.0,0.000000,0.0,M+3,A,Valid,True,M+3,Foods
3,D111_709567,D111,709567,SAF OATS 400G(FRE WT SAF OATS 1KG P)-N,SAFF OATS,2026-03-31,2026-07-31,NaN,0,0,...,0.0,0.0,0.000000,0.0,M+4,A,Valid,True,M+4,Foods
4,D111_718288,D111,718288,SAFF GOLD 5L JAR,SAFF GOLD,2026-03-31,2026-04-30,43.757146,0,0,...,0.0,0.0,0.940128,0.0,M+1,A,Valid,True,M+1,Saffola Oils
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39327,D677_811169,D677,811169,SW MINI EDP GIFT PO4 18ML,SW_SGPRF,2026-03-31,2026-07-31,0.000000,0,0,...,NaN,NaN,0.000000,0.0,M+4,C,Valid,True,M+4,Male Grooming
39328,D677_811181,D677,811181,SAFFOLA COLDPRESS CNO 1L,SAF_CDPRS,2026-03-31,2026-04-30,0.003355,0,0,...,NaN,NaN,0.129492,0.0,M+1,C,Valid,True,M+1,Saffola Oils
39329,D677_811181,D677,811181,SAFFOLA COLDPRESS CNO 1L,SAF_CDPRS,2026-03-31,2026-05-31,0.004645,0,0,...,NaN,NaN,0.129492,0.0,M+2,C,Valid,True,M+2,Saffola Oils
39330,D677_811181,D677,811181,SAFFOLA COLDPRESS CNO 1L,SAF_CDPRS,2026-03-31,2026-06-30,0.004355,0,0,...,NaN,NaN,0.129492,0.0,M+3,C,Valid,True,M+3,Saffola Oils


In [65]:
qcom_df = qcom_df.groupby(['Month Date', 'Depot', 'PSKU',
       'Brand', 'portfolio','M month','Run Month'])[['Calculated Primary Vol']].sum().reset_index()
qcom_df

,Month Date,Depot,PSKU,Brand,portfolio,M month,Run Month,Calculated Primary Vol
0,2026-04-30,D111,709567,SAFF OATS,Foods,M+1,2026-03-31,0.0
1,2026-04-30,D111,718288,SAFF GOLD,Saffola Oils,M+1,2026-03-31,0.0
2,2026-04-30,D111,718297,PCNO(R),CNO,M+1,2026-03-31,0.0
3,2026-04-30,D111,718299,PCNO(R),CNO,M+1,2026-03-31,0.0
4,2026-04-30,D111,718303,REV.LQDST,Others,M+1,2026-03-31,0.0
...,...,...,...,...,...,...,...,...
39327,2026-07-31,D677,810805,PABABY_GM,Skin Care,M+4,2026-03-31,0.0
39328,2026-07-31,D677,810807,PABABY_GM,Skin Care,M+4,2026-03-31,0.0
39329,2026-07-31,D677,810919,PABABY_GM,Skin Care,M+4,2026-03-31,0.0
39330,2026-07-31,D677,811169,SW_SGPRF,Male Grooming,M+4,2026-03-31,0.0


In [66]:

qcom_df.rename(columns = {'Month Date':'Month', 'Final PSKU':'PSKU', 'Depot Code':'Depot', 'Run Month':'run_month','portfolio':'Portfolio'}, inplace = True)
qcom_df

,Month,Depot,PSKU,Brand,Portfolio,M month,run_month,Calculated Primary Vol
0,2026-04-30,D111,709567,SAFF OATS,Foods,M+1,2026-03-31,0.0
1,2026-04-30,D111,718288,SAFF GOLD,Saffola Oils,M+1,2026-03-31,0.0
2,2026-04-30,D111,718297,PCNO(R),CNO,M+1,2026-03-31,0.0
3,2026-04-30,D111,718299,PCNO(R),CNO,M+1,2026-03-31,0.0
4,2026-04-30,D111,718303,REV.LQDST,Others,M+1,2026-03-31,0.0
...,...,...,...,...,...,...,...,...
39327,2026-07-31,D677,810805,PABABY_GM,Skin Care,M+4,2026-03-31,0.0
39328,2026-07-31,D677,810807,PABABY_GM,Skin Care,M+4,2026-03-31,0.0
39329,2026-07-31,D677,810919,PABABY_GM,Skin Care,M+4,2026-03-31,0.0
39330,2026-07-31,D677,811169,SW_SGPRF,Male Grooming,M+4,2026-03-31,0.0


In [67]:
qcom_df[qcom_df['Month'] == '2026-04-30']['Calculated Primary Vol'].sum()

306981.0882611541

In [68]:
qcom_df['Month'] = qcom_df['Month'].astype(str)
qcom_df['run_month'] = qcom_df['run_month'].astype(str)
qcom_df.columns

Index(['Month', 'Depot', 'PSKU', 'Brand', 'Portfolio', 'M month', 'run_month',
       'Calculated Primary Vol'],
      dtype='object')

In [69]:
qcom_df

,Month,Depot,PSKU,Brand,Portfolio,M month,run_month,Calculated Primary Vol
0,2026-04-30,D111,709567,SAFF OATS,Foods,M+1,2026-03-31,0.0
1,2026-04-30,D111,718288,SAFF GOLD,Saffola Oils,M+1,2026-03-31,0.0
2,2026-04-30,D111,718297,PCNO(R),CNO,M+1,2026-03-31,0.0
3,2026-04-30,D111,718299,PCNO(R),CNO,M+1,2026-03-31,0.0
4,2026-04-30,D111,718303,REV.LQDST,Others,M+1,2026-03-31,0.0
...,...,...,...,...,...,...,...,...
39327,2026-07-31,D677,810805,PABABY_GM,Skin Care,M+4,2026-03-31,0.0
39328,2026-07-31,D677,810807,PABABY_GM,Skin Care,M+4,2026-03-31,0.0
39329,2026-07-31,D677,810919,PABABY_GM,Skin Care,M+4,2026-03-31,0.0
39330,2026-07-31,D677,811169,SW_SGPRF,Male Grooming,M+4,2026-03-31,0.0


In [70]:
upload_df = qcom_df[['Month', 'Depot', 'PSKU', 'Brand', 'Portfolio', 'M month', 'run_month',
       'Calculated Primary Vol']]


In [71]:
upload_df['Channel'] = 'Ecom'

In [72]:
upload_df

,Month,Depot,PSKU,Brand,Portfolio,M month,run_month,Calculated Primary Vol,Channel
0,2026-04-30,D111,709567,SAFF OATS,Foods,M+1,2026-03-31,0.0,Ecom
1,2026-04-30,D111,718288,SAFF GOLD,Saffola Oils,M+1,2026-03-31,0.0,Ecom
2,2026-04-30,D111,718297,PCNO(R),CNO,M+1,2026-03-31,0.0,Ecom
3,2026-04-30,D111,718299,PCNO(R),CNO,M+1,2026-03-31,0.0,Ecom
4,2026-04-30,D111,718303,REV.LQDST,Others,M+1,2026-03-31,0.0,Ecom
...,...,...,...,...,...,...,...,...,...
39327,2026-07-31,D677,810805,PABABY_GM,Skin Care,M+4,2026-03-31,0.0,Ecom
39328,2026-07-31,D677,810807,PABABY_GM,Skin Care,M+4,2026-03-31,0.0,Ecom
39329,2026-07-31,D677,810919,PABABY_GM,Skin Care,M+4,2026-03-31,0.0,Ecom
39330,2026-07-31,D677,811169,SW_SGPRF,Male Grooming,M+4,2026-03-31,0.0,Ecom


In [73]:
upload_df.columns = upload_df.columns.str.upper()
upload_df

,MONTH,DEPOT,PSKU,BRAND,PORTFOLIO,M MONTH,RUN_MONTH,CALCULATED PRIMARY VOL,CHANNEL
0,2026-04-30,D111,709567,SAFF OATS,Foods,M+1,2026-03-31,0.0,Ecom
1,2026-04-30,D111,718288,SAFF GOLD,Saffola Oils,M+1,2026-03-31,0.0,Ecom
2,2026-04-30,D111,718297,PCNO(R),CNO,M+1,2026-03-31,0.0,Ecom
3,2026-04-30,D111,718299,PCNO(R),CNO,M+1,2026-03-31,0.0,Ecom
4,2026-04-30,D111,718303,REV.LQDST,Others,M+1,2026-03-31,0.0,Ecom
...,...,...,...,...,...,...,...,...,...
39327,2026-07-31,D677,810805,PABABY_GM,Skin Care,M+4,2026-03-31,0.0,Ecom
39328,2026-07-31,D677,810807,PABABY_GM,Skin Care,M+4,2026-03-31,0.0,Ecom
39329,2026-07-31,D677,810919,PABABY_GM,Skin Care,M+4,2026-03-31,0.0,Ecom
39330,2026-07-31,D677,811169,SW_SGPRF,Male Grooming,M+4,2026-03-31,0.0,Ecom


In [74]:
# push data to snowflake
from snowflake.connector.pandas_tools import write_pandas

write_pandas(dev_conn, upload_df, 
            table_name = "TRN_MIL_DF_OFT2PRIM_SHARED",
            auto_create_table=True,
            overwrite = False,)

(True,
 1,
 39332,
 [('dxodycxita/file0.txt',
   'LOADED',
   39332,
   39332,
   1,
   0,
   None,
   None,
   None,
   None)])

### Push chain psku primary

In [89]:
qcom_df = pd.read_excel(
    '/data/aman_singh/acuuracy_check/ECOM Chain PSKU Primary_as_on_11th_Feb_2026.xlsb',
    sheet_name='Base'
)

In [90]:
qcom_df

,Key,Chain,PSKU,PSKU Desc,Brand,Index Rate,Portfolio,Run Month,Month Date,M month,...,Offtake Actuals Lag 3 Val,LY Offtake Actuals Lag 1 Val,LY Offtake Actuals Lag 2 Val,LY Offtake Actuals Lag 3 Val,LY Offtake Actuals Lead 1 Val,LY Offtake Actuals Lead 2 Val,Calculated Primary Val,Brand Class,Planning Principle,Primary P3M 0?
0,Amazon ARIPL_718287,Amazon ARIPL,718287,PCNO 200ml JAR,PCNO(R),309765.865129,CNO,46081,46112,M+1,...,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000,A,Valid,True
1,Amazon ARIPL_718287,Amazon ARIPL,718287,PCNO 200ml JAR,PCNO(R),309765.865129,CNO,46081,46142,M+2,...,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000,A,Valid,True
2,Amazon ARIPL_718287,Amazon ARIPL,718287,PCNO 200ml JAR,PCNO(R),309765.865129,CNO,46081,46173,M+3,...,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000,A,Valid,True
3,Amazon ARIPL_718287,Amazon ARIPL,718287,PCNO 200ml JAR,PCNO(R),309765.865129,CNO,46081,46203,M+4,...,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000,A,Valid,True
4,Amazon ARIPL_718288,Amazon ARIPL,718288,SAFF GOLD 5L JAR,SAFF GOLD,137662.938527,Saffola Oils,46081,46112,M+1,...,0.28265,0.165361,0.190966,0.22004,0.159827,0.200795,0.34632,A,Valid,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11487,Nykaa_811169,Nykaa,811169,SW MINI EDP GIFT PO4 18ML,SW_SGPRF,1443.400363,Male Grooming,46081,46203,M+4,...,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000,C,Valid,True
11488,Nykaa_811181,Nykaa,811181,SAFFOLA COLDPRESS CNO 1L,SAF_CDPRS,330000.000000,Saffola Oils,46081,46112,M+1,...,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000,C,Valid,True
11489,Nykaa_811181,Nykaa,811181,SAFFOLA COLDPRESS CNO 1L,SAF_CDPRS,330000.000000,Saffola Oils,46081,46142,M+2,...,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000,C,Valid,True
11490,Nykaa_811181,Nykaa,811181,SAFFOLA COLDPRESS CNO 1L,SAF_CDPRS,330000.000000,Saffola Oils,46081,46173,M+3,...,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000,C,Valid,True


In [91]:
qcom_df['Month Date'] = (
    pd.to_datetime(qcom_df['Month Date'], unit='D', origin='1899-12-30')
      .dt.to_period('M')
      .dt.to_timestamp()
)
qcom_df

,Key,Chain,PSKU,PSKU Desc,Brand,Index Rate,Portfolio,Run Month,Month Date,M month,...,Offtake Actuals Lag 3 Val,LY Offtake Actuals Lag 1 Val,LY Offtake Actuals Lag 2 Val,LY Offtake Actuals Lag 3 Val,LY Offtake Actuals Lead 1 Val,LY Offtake Actuals Lead 2 Val,Calculated Primary Val,Brand Class,Planning Principle,Primary P3M 0?
0,Amazon ARIPL_718287,Amazon ARIPL,718287,PCNO 200ml JAR,PCNO(R),309765.865129,CNO,46081,2026-03-01,M+1,...,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000,A,Valid,True
1,Amazon ARIPL_718287,Amazon ARIPL,718287,PCNO 200ml JAR,PCNO(R),309765.865129,CNO,46081,2026-04-01,M+2,...,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000,A,Valid,True
2,Amazon ARIPL_718287,Amazon ARIPL,718287,PCNO 200ml JAR,PCNO(R),309765.865129,CNO,46081,2026-05-01,M+3,...,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000,A,Valid,True
3,Amazon ARIPL_718287,Amazon ARIPL,718287,PCNO 200ml JAR,PCNO(R),309765.865129,CNO,46081,2026-06-01,M+4,...,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000,A,Valid,True
4,Amazon ARIPL_718288,Amazon ARIPL,718288,SAFF GOLD 5L JAR,SAFF GOLD,137662.938527,Saffola Oils,46081,2026-03-01,M+1,...,0.28265,0.165361,0.190966,0.22004,0.159827,0.200795,0.34632,A,Valid,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11487,Nykaa_811169,Nykaa,811169,SW MINI EDP GIFT PO4 18ML,SW_SGPRF,1443.400363,Male Grooming,46081,2026-06-01,M+4,...,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000,C,Valid,True
11488,Nykaa_811181,Nykaa,811181,SAFFOLA COLDPRESS CNO 1L,SAF_CDPRS,330000.000000,Saffola Oils,46081,2026-03-01,M+1,...,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000,C,Valid,True
11489,Nykaa_811181,Nykaa,811181,SAFFOLA COLDPRESS CNO 1L,SAF_CDPRS,330000.000000,Saffola Oils,46081,2026-04-01,M+2,...,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000,C,Valid,True
11490,Nykaa_811181,Nykaa,811181,SAFFOLA COLDPRESS CNO 1L,SAF_CDPRS,330000.000000,Saffola Oils,46081,2026-05-01,M+3,...,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000,C,Valid,True


In [92]:
qcom_df['Month Date'] = (
    pd.to_datetime(qcom_df['Month Date']) + pd.offsets.MonthEnd(0)
)

In [93]:
qcom_df['Run Month'] = (
    pd.to_datetime(qcom_df['Run Month'], unit='D', origin='1899-12-30')
      .dt.to_period('M')
      .dt.to_timestamp()
)
qcom_df['Run Month'] = (
    pd.to_datetime(qcom_df['Run Month']) + pd.offsets.MonthEnd(0)
)
qcom_df

,Key,Chain,PSKU,PSKU Desc,Brand,Index Rate,Portfolio,Run Month,Month Date,M month,...,Offtake Actuals Lag 3 Val,LY Offtake Actuals Lag 1 Val,LY Offtake Actuals Lag 2 Val,LY Offtake Actuals Lag 3 Val,LY Offtake Actuals Lead 1 Val,LY Offtake Actuals Lead 2 Val,Calculated Primary Val,Brand Class,Planning Principle,Primary P3M 0?
0,Amazon ARIPL_718287,Amazon ARIPL,718287,PCNO 200ml JAR,PCNO(R),309765.865129,CNO,2026-02-28,2026-03-31,M+1,...,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000,A,Valid,True
1,Amazon ARIPL_718287,Amazon ARIPL,718287,PCNO 200ml JAR,PCNO(R),309765.865129,CNO,2026-02-28,2026-04-30,M+2,...,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000,A,Valid,True
2,Amazon ARIPL_718287,Amazon ARIPL,718287,PCNO 200ml JAR,PCNO(R),309765.865129,CNO,2026-02-28,2026-05-31,M+3,...,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000,A,Valid,True
3,Amazon ARIPL_718287,Amazon ARIPL,718287,PCNO 200ml JAR,PCNO(R),309765.865129,CNO,2026-02-28,2026-06-30,M+4,...,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000,A,Valid,True
4,Amazon ARIPL_718288,Amazon ARIPL,718288,SAFF GOLD 5L JAR,SAFF GOLD,137662.938527,Saffola Oils,2026-02-28,2026-03-31,M+1,...,0.28265,0.165361,0.190966,0.22004,0.159827,0.200795,0.34632,A,Valid,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11487,Nykaa_811169,Nykaa,811169,SW MINI EDP GIFT PO4 18ML,SW_SGPRF,1443.400363,Male Grooming,2026-02-28,2026-06-30,M+4,...,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000,C,Valid,True
11488,Nykaa_811181,Nykaa,811181,SAFFOLA COLDPRESS CNO 1L,SAF_CDPRS,330000.000000,Saffola Oils,2026-02-28,2026-03-31,M+1,...,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000,C,Valid,True
11489,Nykaa_811181,Nykaa,811181,SAFFOLA COLDPRESS CNO 1L,SAF_CDPRS,330000.000000,Saffola Oils,2026-02-28,2026-04-30,M+2,...,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000,C,Valid,True
11490,Nykaa_811181,Nykaa,811181,SAFFOLA COLDPRESS CNO 1L,SAF_CDPRS,330000.000000,Saffola Oils,2026-02-28,2026-05-31,M+3,...,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000,C,Valid,True


In [94]:
qcom_df['Run Month'] = pd.to_datetime(qcom_df['Run Month'])
qcom_df['Month Date'] = pd.to_datetime(qcom_df['Month Date'])

mappings ={}

for run_month in qcom_df['Run Month'].unique():
    # run_month = pd.to_datetime(run_month)

    if not run_month in mappings:
        mappings[run_month] = {}

    mappings[run_month][run_month] = 'M'

    for idx in range(1, 9):
        mappings[run_month][run_month + MonthEnd(idx)] = f"M+{idx}"


   
qcom_df['M month'] = qcom_df.apply(
    lambda x: mappings[x['Run Month']].get(
        x['Month Date']
    ), axis=1
)

In [ ]:
# qcom_df = qcom_df[qcom_df['M month'] == 'M+1']
# qcom_df

,key2,key,Chain,FC,PSKU,PSKU Desc,Brand,Index Rate,Portfolio,Run Month,...,Offtake Chain FC PSKU Lag 3 Vol,LY Offtake Actuals Chain FC PSKU Vol,LY Offtake Chain FC PSKU P3M Vol,Offtake Chain FC PSKU P3M Vol,Offtake Chain FC PSKU Lag 1 Val,Offtake Chain FC PSKU Lag 2 Val,Offtake Chain FC PSKU Lag 3 Val,LY Offtake Actuals Chain FC PSKU Val,LY Offtake Chain FC PSKU P3M Val,Offtake Chain FC PSKU P3M Val
2,Blinkit_ahmedabad a2 - feeder warehouse_718288,Blinkit_718288,Blinkit,ahmedabad a2 - feeder warehouse,718288,SAFF GOLD 5L JAR,SAFF GOLD,137662.938527,Saffola Oils,2025-11-30,...,6.873931,5.219777,3.804556,6.602009,0.107562,0.085510,0.094629,0.071857,0.052375,0.090885
9,Blinkit_ahmedabad a2 - feeder warehouse_718299,Blinkit_718299,Blinkit,ahmedabad a2 - feeder warehouse,718299,PCNO 100ml BTL,PCNO(R),309765.865129,CNO,2025-11-30,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
16,Blinkit_ahmedabad a2 - feeder warehouse_718308,Blinkit_718308,Blinkit,ahmedabad a2 - feeder warehouse,718308,PCNO 100ml JAR,PCNO(R),309765.865129,CNO,2025-11-30,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
23,Blinkit_ahmedabad a2 - feeder warehouse_718310,Blinkit_718310,Blinkit,ahmedabad a2 - feeder warehouse,718310,PCNO 500ml JAR,PCNO(R),309765.865129,CNO,2025-11-30,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
30,Blinkit_ahmedabad a2 - feeder warehouse_718312,Blinkit_718312,Blinkit,ahmedabad a2 - feeder warehouse,718312,PCNO 1L JAR,PCNO(R),309765.865129,CNO,2025-11-30,...,0.196489,0.114393,0.065902,0.201895,0.006292,0.005917,0.006087,0.003543,0.002041,0.006254
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
235825,Zepto_pun-dry-mh2-koregaon_810673,Zepto_810673,Zepto,pun-dry-mh2-koregaon,810673,PA ESS ROSEMARY OIL 14ML,PA_ESS_HO,12900.000000,Hair Oils,2025-11-30,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
235832,Zepto_pun-dry-mh2-koregaon_810674,Zepto_810674,Zepto,pun-dry-mh2-koregaon,810674,PA ESS TEA TREE OIL 14ML,PA_ESS_HO,12900.000000,Hair Oils,2025-11-30,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
235839,Zepto_pun-dry-mh2-koregaon_810685,Zepto_810685,Zepto,pun-dry-mh2-koregaon,810685,SF MUESLI MANGO 400G POUCH,SAF-MUSLI,321959.667548,Foods,2025-11-30,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
235846,Zepto_pun-dry-mh2-koregaon_810738,Zepto_810738,Zepto,pun-dry-mh2-koregaon,810738,PA BABY FACE BODY WIPE 362GM,PABABY_GM,451.133000,Skin Care,2025-11-30,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [95]:
qcom_df.columns

Index(['Key', 'Chain', 'PSKU', 'PSKU Desc', 'Brand', 'Index Rate', 'Portfolio',
       'Run Month', 'Month Date', 'M month', 'Primary Till Date Actuals Vol',
       'Secondary Plan Vol', 'Primary P3M Vol', 'Offtake Chain PSKU Vol',
       'Offtake Chain PSKU Forecast Vol', 'Norms SOH', 'Norm Days',
       'Safety Stock Vol', 'Actual Closing SOH Vol',
       'Actual Closing SOH Lag 1 Vol', 'Actual Closing SOH Lag 2 Vol',
       'Assumed Closing SOH Vol', 'Assumed Closing SOH Lag 1 Vol',
       'Final Assumed Closing SOH Vol', 'Final Assumed Closing SOH Lag 1 Vol',
       'Primary Actuals Vol', 'Sec Actuals Vol', 'Primary P3M redundant Vol',
       'LY Primary Actuals Vol', 'LY Sec Actuals Vol', 'LY Primary P3M Vol',
       'Primary Actuals Lag 1 Vol', 'Primary Actuals Lag 2 Vol',
       'Primary Actuals Lag 3 Vol', 'LY Primary Actuals Lag 1 Vol',
       'LY Primary Actuals Lag 2 Vol', 'LY Primary Actuals Lag 3 Vol',
       'LY Primary Actuals Lead 1 Vol', 'LY Primary Actuals Lead 2 Vol'

In [96]:
qcom_df.rename(columns = {'Calculated Depot PSKU Primary Vol':'Calculated Primary Vol'}, inplace = True)

In [97]:
brand_md_df = pd.read_excel(r"/data/aman_singh/mt_forecast/Brand_metadata.xlsx")
brand_md_df.rename(columns = {'brand_code':'Brand'}, inplace = True)

In [98]:

len_before_merge = len(qcom_df)
qcom_df = qcom_df.merge(
    brand_md_df,
    on=['Brand'],
    how='left'
)
assert len_before_merge == len(qcom_df)

In [99]:
qcom_df

,Key,Chain,PSKU,PSKU Desc,Brand,Index Rate,Portfolio,Run Month,Month Date,M month,...,LY Offtake Actuals Lag 1 Val,LY Offtake Actuals Lag 2 Val,LY Offtake Actuals Lag 3 Val,LY Offtake Actuals Lead 1 Val,LY Offtake Actuals Lead 2 Val,Calculated Primary Val,Brand Class,Planning Principle,Primary P3M 0?,portfolio
0,Amazon ARIPL_718287,Amazon ARIPL,718287,PCNO 200ml JAR,PCNO(R),309765.865129,CNO,2026-02-28,2026-03-31,M+1,...,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000,A,Valid,True,CNO
1,Amazon ARIPL_718287,Amazon ARIPL,718287,PCNO 200ml JAR,PCNO(R),309765.865129,CNO,2026-02-28,2026-04-30,M+2,...,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000,A,Valid,True,CNO
2,Amazon ARIPL_718287,Amazon ARIPL,718287,PCNO 200ml JAR,PCNO(R),309765.865129,CNO,2026-02-28,2026-05-31,M+3,...,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000,A,Valid,True,CNO
3,Amazon ARIPL_718287,Amazon ARIPL,718287,PCNO 200ml JAR,PCNO(R),309765.865129,CNO,2026-02-28,2026-06-30,M+4,...,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000,A,Valid,True,CNO
4,Amazon ARIPL_718288,Amazon ARIPL,718288,SAFF GOLD 5L JAR,SAFF GOLD,137662.938527,Saffola Oils,2026-02-28,2026-03-31,M+1,...,0.165361,0.190966,0.22004,0.159827,0.200795,0.34632,A,Valid,False,Saffola Oils
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11487,Nykaa_811169,Nykaa,811169,SW MINI EDP GIFT PO4 18ML,SW_SGPRF,1443.400363,Male Grooming,2026-02-28,2026-06-30,M+4,...,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000,C,Valid,True,Male Grooming
11488,Nykaa_811181,Nykaa,811181,SAFFOLA COLDPRESS CNO 1L,SAF_CDPRS,330000.000000,Saffola Oils,2026-02-28,2026-03-31,M+1,...,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000,C,Valid,True,Saffola Oils
11489,Nykaa_811181,Nykaa,811181,SAFFOLA COLDPRESS CNO 1L,SAF_CDPRS,330000.000000,Saffola Oils,2026-02-28,2026-04-30,M+2,...,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000,C,Valid,True,Saffola Oils
11490,Nykaa_811181,Nykaa,811181,SAFFOLA COLDPRESS CNO 1L,SAF_CDPRS,330000.000000,Saffola Oils,2026-02-28,2026-05-31,M+3,...,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000,C,Valid,True,Saffola Oils


In [101]:
qcom_df = qcom_df.groupby(['Month Date', 'Chain', 'PSKU',
       'Brand', 'portfolio','M month','Run Month'])[['Calculated Primary Vol']].sum().reset_index()
qcom_df

,Month Date,Chain,PSKU,Brand,portfolio,M month,Run Month,Calculated Primary Vol
0,2026-03-31,Amazon ARIPL,718287,PCNO(R),CNO,M+1,2026-02-28,0.000000
1,2026-03-31,Amazon ARIPL,718288,SAFF GOLD,Saffola Oils,M+1,2026-02-28,25.157119
2,2026-03-31,Amazon ARIPL,718297,PCNO(R),CNO,M+1,2026-02-28,0.000000
3,2026-03-31,Amazon ARIPL,718299,PCNO(R),CNO,M+1,2026-02-28,0.000000
4,2026-03-31,Amazon ARIPL,718308,PCNO(R),CNO,M+1,2026-02-28,0.000000
...,...,...,...,...,...,...,...,...
11487,2026-06-30,Nykaa,810805,PABABY_GM,Skin Care,M+4,2026-02-28,0.000000
11488,2026-06-30,Nykaa,810807,PABABY_GM,Skin Care,M+4,2026-02-28,0.000000
11489,2026-06-30,Nykaa,810919,PABABY_GM,Skin Care,M+4,2026-02-28,0.000000
11490,2026-06-30,Nykaa,811169,SW_SGPRF,Male Grooming,M+4,2026-02-28,0.000000


In [102]:

qcom_df.rename(columns = {'Month Date':'Month', 'Final PSKU':'PSKU', 'Depot Code':'Depot', 'Run Month':'run_month','portfolio':'Portfolio'}, inplace = True)
qcom_df

,Month,Chain,PSKU,Brand,Portfolio,M month,run_month,Calculated Primary Vol
0,2026-03-31,Amazon ARIPL,718287,PCNO(R),CNO,M+1,2026-02-28,0.000000
1,2026-03-31,Amazon ARIPL,718288,SAFF GOLD,Saffola Oils,M+1,2026-02-28,25.157119
2,2026-03-31,Amazon ARIPL,718297,PCNO(R),CNO,M+1,2026-02-28,0.000000
3,2026-03-31,Amazon ARIPL,718299,PCNO(R),CNO,M+1,2026-02-28,0.000000
4,2026-03-31,Amazon ARIPL,718308,PCNO(R),CNO,M+1,2026-02-28,0.000000
...,...,...,...,...,...,...,...,...
11487,2026-06-30,Nykaa,810805,PABABY_GM,Skin Care,M+4,2026-02-28,0.000000
11488,2026-06-30,Nykaa,810807,PABABY_GM,Skin Care,M+4,2026-02-28,0.000000
11489,2026-06-30,Nykaa,810919,PABABY_GM,Skin Care,M+4,2026-02-28,0.000000
11490,2026-06-30,Nykaa,811169,SW_SGPRF,Male Grooming,M+4,2026-02-28,0.000000


In [103]:
qcom_df[qcom_df['Month'] == '2026-03-31']['Calculated Primary Vol'].sum()

258841.9104424352

In [104]:
qcom_df['Month'] = qcom_df['Month'].astype(str)
qcom_df['run_month'] = qcom_df['run_month'].astype(str)
qcom_df.columns

Index(['Month', 'Chain', 'PSKU', 'Brand', 'Portfolio', 'M month', 'run_month',
       'Calculated Primary Vol'],
      dtype='object')

In [105]:
qcom_df

,Month,Chain,PSKU,Brand,Portfolio,M month,run_month,Calculated Primary Vol
0,2026-03-31,Amazon ARIPL,718287,PCNO(R),CNO,M+1,2026-02-28,0.000000
1,2026-03-31,Amazon ARIPL,718288,SAFF GOLD,Saffola Oils,M+1,2026-02-28,25.157119
2,2026-03-31,Amazon ARIPL,718297,PCNO(R),CNO,M+1,2026-02-28,0.000000
3,2026-03-31,Amazon ARIPL,718299,PCNO(R),CNO,M+1,2026-02-28,0.000000
4,2026-03-31,Amazon ARIPL,718308,PCNO(R),CNO,M+1,2026-02-28,0.000000
...,...,...,...,...,...,...,...,...
11487,2026-06-30,Nykaa,810805,PABABY_GM,Skin Care,M+4,2026-02-28,0.000000
11488,2026-06-30,Nykaa,810807,PABABY_GM,Skin Care,M+4,2026-02-28,0.000000
11489,2026-06-30,Nykaa,810919,PABABY_GM,Skin Care,M+4,2026-02-28,0.000000
11490,2026-06-30,Nykaa,811169,SW_SGPRF,Male Grooming,M+4,2026-02-28,0.000000


In [106]:
upload_df = qcom_df[['Month', 'Chain', 'PSKU', 'Brand', 'Portfolio', 'M month', 'run_month',
       'Calculated Primary Vol']]


In [107]:
upload_df['Channel'] = 'Ecom'

In [108]:
upload_df

,Month,Chain,PSKU,Brand,Portfolio,M month,run_month,Calculated Primary Vol,Channel
0,2026-03-31,Amazon ARIPL,718287,PCNO(R),CNO,M+1,2026-02-28,0.000000,Ecom
1,2026-03-31,Amazon ARIPL,718288,SAFF GOLD,Saffola Oils,M+1,2026-02-28,25.157119,Ecom
2,2026-03-31,Amazon ARIPL,718297,PCNO(R),CNO,M+1,2026-02-28,0.000000,Ecom
3,2026-03-31,Amazon ARIPL,718299,PCNO(R),CNO,M+1,2026-02-28,0.000000,Ecom
4,2026-03-31,Amazon ARIPL,718308,PCNO(R),CNO,M+1,2026-02-28,0.000000,Ecom
...,...,...,...,...,...,...,...,...,...
11487,2026-06-30,Nykaa,810805,PABABY_GM,Skin Care,M+4,2026-02-28,0.000000,Ecom
11488,2026-06-30,Nykaa,810807,PABABY_GM,Skin Care,M+4,2026-02-28,0.000000,Ecom
11489,2026-06-30,Nykaa,810919,PABABY_GM,Skin Care,M+4,2026-02-28,0.000000,Ecom
11490,2026-06-30,Nykaa,811169,SW_SGPRF,Male Grooming,M+4,2026-02-28,0.000000,Ecom


In [109]:
upload_df.columns = upload_df.columns.str.upper()
upload_df

,MONTH,CHAIN,PSKU,BRAND,PORTFOLIO,M MONTH,RUN_MONTH,CALCULATED PRIMARY VOL,CHANNEL
0,2026-03-31,Amazon ARIPL,718287,PCNO(R),CNO,M+1,2026-02-28,0.000000,Ecom
1,2026-03-31,Amazon ARIPL,718288,SAFF GOLD,Saffola Oils,M+1,2026-02-28,25.157119,Ecom
2,2026-03-31,Amazon ARIPL,718297,PCNO(R),CNO,M+1,2026-02-28,0.000000,Ecom
3,2026-03-31,Amazon ARIPL,718299,PCNO(R),CNO,M+1,2026-02-28,0.000000,Ecom
4,2026-03-31,Amazon ARIPL,718308,PCNO(R),CNO,M+1,2026-02-28,0.000000,Ecom
...,...,...,...,...,...,...,...,...,...
11487,2026-06-30,Nykaa,810805,PABABY_GM,Skin Care,M+4,2026-02-28,0.000000,Ecom
11488,2026-06-30,Nykaa,810807,PABABY_GM,Skin Care,M+4,2026-02-28,0.000000,Ecom
11489,2026-06-30,Nykaa,810919,PABABY_GM,Skin Care,M+4,2026-02-28,0.000000,Ecom
11490,2026-06-30,Nykaa,811169,SW_SGPRF,Male Grooming,M+4,2026-02-28,0.000000,Ecom


In [110]:
# push data to snowflake
from snowflake.connector.pandas_tools import write_pandas

write_pandas(dev_conn, upload_df, 
            table_name = "TRN_MIL_DF_OFT2PRIM_CPSKU",
            auto_create_table=True,
            overwrite = False,)

(True,
 1,
 11492,
 [('djkwapqmrb/file0.txt',
   'LOADED',
   11492,
   11492,
   1,
   0,
   None,
   None,
   None,
   None)])

### push offtakes data

In [86]:
offtakes_df = pd.read_excel('/data/aman_singh/acuuracy_check/offtakes/Heuristics_all_combination_qcom_cp_mar_live.xlsx', sheet_name = 'Base')


In [87]:
offtakes_df.columns[50:]

Index(['OT_Value_in_Cr_lag_1', 'OT_Value_in_Cr_lag_2', 'OT_Value_in_Cr_lag_3',
       'class', 'skipped', 'seasonality_flag', 'final_trend',
       'lower_threshold', 'upper_threshold', 'recency_factor',
       'shrink_ratio_prophet', 'shrink_ratio_rf', 'p3m_ly_growth',
       'recency_heuristic_prophet_vol', 'seasonal_heuristic_prophet_vol',
       'rec_seas_heuristic_prophet_vol', 'non_seasonal_heuristic_prophet_vol',
       'final_heuristic_prophet_vol', 'final_heuristic_prophet_value',
       'final_heuristic_prophet_value_2', 'final_heuristic_prophet_vol_2',
       'recency_heuristic_rf_vol', 'seasonal_heuristic_rf_vol',
       'rec_seas_heuristic_prophet_vol.1',
       'non_seasonal_heuristic_prophet_vol.1', 'final_heuristic_rf_vol',
       'final_heuristic_rf_value', 'final_heuristic_rf_value_2',
       'final_heuristic_rf_vol_2', 'error_prophet_vol',
       'abs_error_prophet_vol', 'error_prophet_value',
       'abs_error_prophet_value', 'error_rf_vol', 'abs_error_rf_vol',
    

In [88]:

offtakes_df.rename(columns = {'run_month_x':'run_month'}, inplace = True)
offtakes_df['run_month'].unique()

<DatetimeArray>
['2026-03-31 00:00:00']
Length: 1, dtype: datetime64[ns]

In [89]:
offtakes_df = offtakes_df.groupby(['month_date','platform_name', 'parent_material_code', 'brand_code','portfolio','run_month', 'M month']
                    )[['pred_prophet', 'pred_rf','final_heuristic_60_prophet_value_2']].sum().reset_index()

In [90]:
offtakes_df.rename(columns = {'final_heuristic_60_prophet_value_2':'final_heuristic_prophet_value_2'}, inplace = True)

In [91]:
offtakes_df['month_date'] = offtakes_df['month_date'].astype(str)
offtakes_df['run_month'] = offtakes_df['run_month'].astype(str)
offtakes_df.columns

Index(['month_date', 'platform_name', 'parent_material_code', 'brand_code',
       'portfolio', 'run_month', 'M month', 'pred_prophet', 'pred_rf',
       'final_heuristic_prophet_value_2'],
      dtype='object')

In [92]:
upload_df = offtakes_df.copy()

In [93]:
upload_df

,month_date,platform_name,parent_material_code,brand_code,portfolio,run_month,M month,pred_prophet,pred_rf,final_heuristic_prophet_value_2
0,2026-03-31,Blinkit,718288,SAFF GOLD,Saffola Oils,2026-03-31,M,67.980841,60.532758,0.952817
1,2026-03-31,Blinkit,718310,PCNO(R),CNO,2026-03-31,M,0.000000,0.000000,0.000072
2,2026-03-31,Blinkit,718312,PCNO(R),CNO,2026-03-31,M,5.801935,5.906009,0.190297
3,2026-03-31,Blinkit,718315,PCNO(R),CNO,2026-03-31,M,0.006738,0.002800,0.000000
4,2026-03-31,Blinkit,718317,H&C,Hair Oils,2026-03-31,M,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...
7079,2026-11-30,Zepto,810522,SAF_CDPRS,Saffola Oils,2026-03-31,M+8,0.000000,0.000000,0.003014
7080,2026-11-30,Zepto,810673,PA_ESS_HO,Hair Oils,2026-03-31,M+8,0.000000,0.000000,0.010053
7081,2026-11-30,Zepto,810674,PA_ESS_HO,Hair Oils,2026-03-31,M+8,0.000000,0.000000,0.007922
7082,2026-11-30,Zepto,810685,SAF-MUSLI,Foods,2026-03-31,M+8,0.000000,0.000000,0.000580


In [94]:
upload_df['channel'] = 'QCOM'

In [95]:
upload_df.columns = upload_df.columns.str.upper()
upload_df

,MONTH_DATE,PLATFORM_NAME,PARENT_MATERIAL_CODE,BRAND_CODE,PORTFOLIO,RUN_MONTH,M MONTH,PRED_PROPHET,PRED_RF,FINAL_HEURISTIC_PROPHET_VALUE_2,CHANNEL
0,2026-03-31,Blinkit,718288,SAFF GOLD,Saffola Oils,2026-03-31,M,67.980841,60.532758,0.952817,QCOM
1,2026-03-31,Blinkit,718310,PCNO(R),CNO,2026-03-31,M,0.000000,0.000000,0.000072,QCOM
2,2026-03-31,Blinkit,718312,PCNO(R),CNO,2026-03-31,M,5.801935,5.906009,0.190297,QCOM
3,2026-03-31,Blinkit,718315,PCNO(R),CNO,2026-03-31,M,0.006738,0.002800,0.000000,QCOM
4,2026-03-31,Blinkit,718317,H&C,Hair Oils,2026-03-31,M,0.000000,0.000000,0.000000,QCOM
...,...,...,...,...,...,...,...,...,...,...,...
7079,2026-11-30,Zepto,810522,SAF_CDPRS,Saffola Oils,2026-03-31,M+8,0.000000,0.000000,0.003014,QCOM
7080,2026-11-30,Zepto,810673,PA_ESS_HO,Hair Oils,2026-03-31,M+8,0.000000,0.000000,0.010053,QCOM
7081,2026-11-30,Zepto,810674,PA_ESS_HO,Hair Oils,2026-03-31,M+8,0.000000,0.000000,0.007922,QCOM
7082,2026-11-30,Zepto,810685,SAF-MUSLI,Foods,2026-03-31,M+8,0.000000,0.000000,0.000580,QCOM


In [96]:
# push data to snowflake
from snowflake.connector.pandas_tools import write_pandas

write_pandas(dev_conn, upload_df, 
            table_name = "TRN_MIL_DF_OFFTAKES_OUTPUT",
            auto_create_table=True,
            overwrite = False,)

(True,
 1,
 7084,
 [('xsebobxqqt/file0.txt',
   'LOADED',
   7084,
   7084,
   1,
   0,
   None,
   None,
   None,
   None)])